# Assignment 2: Classification Models and Evaluation Metrics
**Course/Topic:** Model Evaluation, SVM vs. Logistic Regression, and Business Metrics

---

## Overview
In this notebook, we train, compare, and evaluate two prominent classification algorithms:
1. **Logistic Regression** (a probabilistic linear model)
2. **Support Vector Machine (SVM)** (a margin-maximizing classifier)

We will evaluate both models using multiple metrics:
- **Confusion Matrix**
- **Accuracy**
- **Precision**
- **Recall**
- **F1-score**

We will also explore the critical difference between standard training and class-balanced training, which is essential for real-world financial risk forecasting where the target variable is highly imbalanced.

---
## Business Problem Context
We are predicting `High_Investment_Risk` (Yes/No) using the **Financial Customer Investment Risk Dataset**. Missing a high-risk customer (False Negative) has severe financial and legal implications, whereas misidentifying a low-risk customer as high-risk (False Positive) represents an operational inefficiency (wasted advisor time).

---

In [ ]:
# Step 0: Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

# Visual styling
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
import warnings
warnings.filterwarnings('ignore')

## 1. Load and Preprocess the Dataset
We read the dataset, define our features and target variable, perform an 80/20 train-test split stratified on the target, and set up our preprocessing pipeline to impute missing values, scale numerical columns, and one-hot encode categorical features.

In [ ]:
# Load data
df = pd.read_csv('financial_customer_investment_risk_dataset (1).xls')

# Define features and target
X = df.drop(columns=['Customer_ID', 'High_Investment_Risk'])
y = df['High_Investment_Risk'].map({'Yes': 1, 'No': 0})

# Train-Test Split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Identify column types
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

# Define Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), num_cols),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), cat_cols)
    ]
)

## Helper Evaluation Function
To simplify comparisons, we define a helper function to calculate the confusion matrix, accuracy, precision, recall, and F1-score for each model.

In [ ]:
def evaluate_model(pipeline, X_train, y_train, X_test, y_test, model_name):
    # Fit the pipeline
    pipeline.fit(X_train, y_train)
    # Predict
    y_pred = pipeline.predict(X_test)
    
    # Calculate metrics
    cm = confusion_matrix(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    
    # Print results
    print(f"=== {model_name} Results ===")
    print("Confusion Matrix:")
    print(cm)
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-Score:  {f1:.4f}\n")
    
    return {
        'Model': model_name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'Confusion_Matrix': cm
    }

## 2. Model 1: Logistic Regression
We train and evaluate two Logistic Regression models:
1. **Standard Logistic Regression:** No class weighting.
2. **Balanced Logistic Regression:** Uses `class_weight='balanced'` to account for class imbalance.

In [ ]:
# Standard Logistic Regression Pipeline
lr_std_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42))
])
lr_std_results = evaluate_model(lr_std_pipeline, X_train, y_train, X_test, y_test, "Standard Logistic Regression")

# Balanced Logistic Regression Pipeline
lr_bal_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(class_weight='balanced', random_state=42))
])
lr_bal_results = evaluate_model(lr_bal_pipeline, X_train, y_train, X_test, y_test, "Balanced Logistic Regression")

## 3. Model 2: Support Vector Machine (SVM)
We train and evaluate two SVM models (using a linear kernel for fair comparison with Logistic Regression):
1. **Standard SVM:** No class weighting.
2. **Balanced SVM:** Uses `class_weight='balanced'` to account for class imbalance.

In [ ]:
# Standard SVM Pipeline
svm_std_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', SVC(kernel='linear', random_state=42))
])
svm_std_results = evaluate_model(svm_std_pipeline, X_train, y_train, X_test, y_test, "Standard SVM (Linear)")

# Balanced SVM Pipeline
svm_bal_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', SVC(kernel='linear', class_weight='balanced', random_state=42))
])
svm_bal_results = evaluate_model(svm_bal_pipeline, X_train, y_train, X_test, y_test, "Balanced SVM (Linear)")

## 4. Compare the Results
Let's aggregate the evaluation metrics into a pandas DataFrame and visualize them to compare model performances.

In [ ]:
# Collect results
results_list = [lr_std_results, lr_bal_results, svm_std_results, svm_bal_results]
comparison_df = pd.DataFrame(results_list).drop(columns=['Confusion_Matrix'])

print("--- Model Comparison Table ---")
display(comparison_df)

# Reshape data for plotting
comparison_melted = comparison_df.melt(id_vars='Model', var_name='Metric', value_name='Score')

# Bar plot comparison
plt.figure(figsize=(12, 6))
sns.barplot(data=comparison_melted, x='Metric', y='Score', hue='Model', palette='Set2')
plt.title('Comparison of Classification Models and Metrics')
plt.ylim(0, 1.05)
plt.ylabel('Score')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Plotting the Confusion Matrices side-by-side
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()

for i, res in enumerate(results_list):
    disp = ConfusionMatrixDisplay(confusion_matrix=res['Confusion_Matrix'], display_labels=['No Risk (0)', 'High Risk (1)'])
    disp.plot(cmap='Blues', ax=axes[i], values_format='d')
    axes[i].grid(False)
    axes[i].set_title(res['Model'])

plt.tight_layout()
plt.show()

---

## Task 2: Compare the Results and Interpret Them

### 1. Which model performed better?
- **Without Balancing (Baseline):** The **Standard Logistic Regression** and **Standard SVM** both achieved very high accuracies (~93.59% and ~92.31% respectively), but this is a false victory. Since the dataset is extremely imbalanced, both models predicted almost every client as `No Risk (0)`. They both had **0.00 Precision, 0.00 Recall, and 0.00 F1-score** for the high-risk class.
- **With Balancing (`class_weight='balanced'`):** 
  - **Balanced Logistic Regression** achieved an **Accuracy of 92.31%**, a **Recall of 50.00%**, a **Precision of 33.33%**, and an **F1-Score of 40.00%**.
  - **Balanced SVM** achieved an **Accuracy of 91.03%**, a **Recall of 50.00%**, a **Precision of 28.57%**, and an **F1-Score of 36.36%**.
  - **Conclusion:** **Balanced Logistic Regression performed better**. While both balanced models achieved the same recall (50.00%), Logistic Regression achieved higher precision (33.33% vs 28.57%), resulting in a higher F1-score (40.00% vs 36.36%) and higher accuracy (92.31% vs 91.03%). It had fewer false positives (4 vs 5).

### 2. Which metric is most important for your business problem?
- **Recall (Sensitivity) on the High Risk Class** is the most important metric for this business problem.
- **Why?** In investment advising, failing to identify a high-risk customer (False Negative) can have catastrophic consequences. The client might invest in highly volatile assets, suffer extreme losses, and sue the firm, or the firm could violate compliance policies. On the other hand, misidentifying a low-risk customer as high-risk (False Positive) merely leads to the advisor spending extra time reviewing the client's profile or recommending a slightly more conservative portfolio, which is far less damaging. Thus, we want to maximize Recall (minimize False Negatives) even if it leads to a lower Precision.

### 3. What do false positives and false negatives mean in your dataset?
- **False Positive (FP):** The model predicts a customer has **High Investment Risk**, but they actually **do not** (Actual: No, Predicted: Yes).
  - *Business Impact:* The advisor might restrict the customer from making profitable high-yield investments or waste resources doing extra portfolio analysis, potentially annoying a client who wanted more growth-oriented assets.
- **False Negative (FN):** The model predicts a customer does **not** have High Investment Risk, but they actually **do** (Actual: Yes, Predicted: No).
  - *Business Impact:* The customer is allowed to invest in volatile assets that exceed their true risk threshold, leading to devastating financial losses, regulatory non-compliance, and potential legal lawsuits against the advisory firm.

### 4. What is one possible limitation or bias in your model?
- **Class Imbalance & Representation Bias:** The dataset is extremely small (387 rows) and has a major class imbalance (~95% "No" vs ~5% "Yes"). The model is heavily biased toward predicting the majority class because it sees very few examples of high-risk customers. 
- Even with `class_weight='balanced'`, the model still has a high False Positive Rate (low precision of 33.33%), meaning that for every correct high-risk prediction, it makes two false ones. The model is also linear and cannot capture non-linear interactions between age, experience, and portfolio value.

### 5. Why should human judgment still be used?
- **ML Models are Blind to Qualitative Context:** The machine learning model only knows what is in the data columns. It does not know if a client's health is failing, if they are planning to buy a house next month, if they are going through a divorce, or if their industry is suffering a unique downturn.
- **Empathy and Communication:** Financial advisors can have deep conversations with clients to understand their psychological comfort with risk. A model might flag someone as "low risk" based on high income, but that person might be psychologically prone to panic-selling during a market dip.
- **Model Errors:** As seen, the best model has a Recall of 50.00% (misses half of high-risk clients) and Precision of 33.33% (two-thirds of predicted high-risk clients are false alarms). Relying solely on the model is extremely unsafe. The model should be used as a **decision support tool**, not a final decision-maker.